**XLR8R: A lex/syntax assistant**

Presenting $XLR8R$ for building tokenizers and parsers; code name $XLR8R$, a mnemonic for accelerator. Its purpose is to extract structure from files with unknown formats or grammars so that the data can be further analyzed.

The simple tool works based on a feedback-control loop. Previously seen patterns are applied to the input data. As these patterns are recognized in context, new patterns emerge based on what's already been seen, the developer makes small, incremental adjustments by incorporating these new patterns back into the set of seen patterns, and the process repeats until all data is properly organized.

For example, let's bootstrap a calculator-expression parser by repeatedly feeding back into the system the output patterns produced from matching properly formed samples of these expressions. With each cycle, the parser should become more complete and accurate.

In [1]:
*       Nothing to install and nothing to import: patterns are part of the language.
        digits = '0123456789'
        lcase = &LCASE
        ucase = &UCASE
*       An action that runs only when the whole match succeeds is written as a
*       conditional assignment to a function call:   '' . *action()
*       The function does its work and returns a name (NRETURN).

In order to ensure an abundance of properly formed expressions, utilize an expression generator either exhaustively or randomly. For our purpose, a random expression generator should work nicely. The function below, rand_expression, returns a well-formed expression with random length and randomized content.

In [2]:
*       A seeded generator, so that every run of this notebook prints the same lines.
        seed = 2026
        DEFINE('rand(n)')                                       :(rand_end)
rand    seed = REMDR(seed * 69069 + 1, 4294967296)
        rand = REMDR(seed / 65536, n) + 1                       :(RETURN)
rand_end
        DEFINE('rand_item()r')                                  :(rand_item_end)
rand_item r = rand(100)
        rand_item = LE(r,13) 'x'                                :S(RETURN)
        rand_item = LE(r,26) 'y'                                :S(RETURN)
        rand_item = LE(r,39) 'z'                                :S(RETURN)
        rand_item = LE(r,80) rand(17) - 1                       :S(RETURN)
        rand_item = '(' rand_term() ')'                         :(RETURN)
rand_item_end
        DEFINE('rand_element()r')                               :(rand_element_end)
rand_element r = rand(100)
        rand_element = LE(r,90) rand_item()                     :S(RETURN)
        rand_element = LE(r,95) '+' rand_element()              :S(RETURN)
        rand_element = '-' rand_element()                       :(RETURN)
rand_element_end
        DEFINE('rand_factor()r')                                :(rand_factor_end)
rand_factor r = rand(100)
        rand_factor = LE(r,70) rand_element()                   :S(RETURN)
        rand_factor = LE(r,85) rand_element() '*' rand_factor() :S(RETURN)
        rand_factor = rand_element() '/' rand_factor()          :(RETURN)
rand_factor_end
        DEFINE('rand_term()r')                                  :(rand_term_end)
rand_term r = rand(100)
        rand_term = LE(r,70) rand_factor()                      :S(RETURN)
        rand_term = LE(r,85) rand_factor() '+' rand_term()      :S(RETURN)
        rand_term = rand_factor() '-' rand_term()               :(RETURN)
rand_term_end
        DEFINE('rand_expression()')                             :(rand_expression_end)
rand_expression rand_expression = rand_term()                   :(RETURN)
rand_expression_end

One of the very first steps is to determine the usage of characters within the unknown input, i.e. the set of characters and frequency of use.

In [3]:
        chars = TABLE()
        k = 0
c5next  k = LT(k,1024) k + 1                                    :F(c5show)
        text = rand_expression()
c5char  text LEN(1) . ch =                                      :F(c5next)
        chars[ch] = chars[ch] + 1                               :(c5char)
c5show  rows = SORT(chars)
        k = 0
c5line  k = k + 1
        OUTPUT = rows[k,1] ' ' rows[k,2]                        :S(c5line)

( 696
) 696
* 558
+ 548
- 623
/ 565
0 167
1 815
2 180
3 189
4 182
5 150
6 159
7 102
8 98
9 101
x 469
y 476
z 466


Another first step is to discover possible tokens. Begin by looking at sequences of traditional classes of characters like digits, upper-case letters, lower-case letters, and special character sequences as operator symbols.

In [4]:
        tokens = TABLE()
        DEFINE('count_token()')                                 :(count_token_end)
count_token tokens[token] = tokens[token] + 1
        count_token = .dummy                                    :(NRETURN)
count_token_end
        token_statistics = POS(0)
+           ARBNO((SPAN(digits) | SPAN(ucase) | SPAN(lcase) | SPAN('+-*/') | LEN(1)) . token
+                 '' . *count_token())
+           RPOS(0)
        k = 0
t7next  k = LT(k,128) k + 1                                     :F(t7show)
        rand_expression() token_statistics                      :(t7next)
t7show  rows = SORT(tokens)
        k = 0
t7line  k = k + 1
        OUTPUT = rows[k,1] ' ' rows[k,2]                        :S(t7line)

( 81
) 81
* 55
*+ 4
*- 1
+ 37
++ 6
+- 1
+-++ 1
- 53
-+ 1
-+- 1
-- 4
/ 50
/+ 3
/- 5
0 7
1 13
10 11
11 10
12 12
13 13
14 11
15 15
16 8
2 4
3 13
4 12
5 7
6 10
7 17
8 8
9 14
x 51
y 42
z 47


In this next code snippet, the $XRL8R\_expressions$ function, is a simple engine that produces a number of random expressions, analyzes each one with the XLR8R pattern, and returns the results of that analysis.

In [5]:
        DEFINE('xlr8r_expressions()cnt,subject')                :(xlr8r_expressions_end)
xlr8r_expressions cnt = 0
xe1     subject = LT(cnt,8) rand_expression()                   :F(RETURN)
        GT(SIZE(subject),10)                                    :S(xe1)
        cnt = cnt + 1
        subject xlr8r('P')                                      :F(xe2)
        OUTPUT = 'True  ' RPAD(subject,10) '  ' P               :(xe1)
xe2     OUTPUT = 'False ' RPAD(subject,10)                      :(xe1)
xlr8r_expressions_end

What follows is the core componenent of the feedback-control mechanism. The translator, $XLR8R$, function returns a PATTERN which recognizes a sequence of known patterns and returns what it recognized. What $XLR8R$ returns is Python code which when compiled and executed creates a PATTERN that matches that specific input. Thus, the XLR8R pattern encodes **proper subsequents** within the context of a particular set of known patterns.

$XLR8R$ produces a PATTERN that matches the string that XLR8R just matched. That PATTERN is a sequence of PATTERNs from the known set of patterns.

The $XLR8R\_expressions$ function then produces a set of varying PATTERNs by iterating over several samples of properly-formed input. Each of these varying PATTERNs recognize properly formed inputs. Thus, the $XLR8R\_expressions$ function encodes the **proper alternates** within the context of a particular set of known patterns.

This $XLR8R$ function will be revised at every iteration of the feedback-control loop by incorporating patterns from the output back into the XLR8R pattern for the next iteration.

In [6]:
        DEFINE('clear(name)')                                   :(clear_end)
clear   $name =
        clear = .dummy                                          :(NRETURN)
clear_end
        DEFINE('add(name,s)')                                   :(add_end)
add     $name = DIFFER($name) $name ' + ' s                     :S(add1)
        $name = s
add1    add = .dummy                                            :(NRETURN)
add_end
        DEFINE('xlr8r(name)','xlr8r11')                        :(xlr8r11_end)
xlr8r11 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( ANY(lcase) '' . *add(xname,'ANY(LCASE)')
+                    | SPAN(digits) '' . *add(xname,'SPAN(DIGITS)')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r11_end
        xlr8r_expressions()

True  +-z         '+' + '-' + ANY(LCASE)
True  2           SPAN(DIGITS)
True  14          SPAN(DIGITS)
True  -8          '-' + SPAN(DIGITS)
True  (9/(x)*z)   '(' + SPAN(DIGITS) + '/' + '(' + ANY(LCASE) + ')' + '*' + ANY(LCASE) + ')'
True  5           SPAN(DIGITS)
True  -7/(0)/y    '-' + SPAN(DIGITS) + '/' + '(' + SPAN(DIGITS) + ')' + '/' + ANY(LCASE)
True  3           SPAN(DIGITS)


Note from the token statistics that we saw spans of digits and single lowercase letters. So within the XLR8R pattern we generate a new pattern $V$ to recognize $ANY(LCASE)$, and pattern $I$ to recognize $SPAN(DIGITS)$. Repeatedly rerun XLR8R to generate varying sets of patterns from random samples.

In [7]:
        V = ANY(lcase)
        I = SPAN(digits)
        DEFINE('xlr8r(name)','xlr8r13')                        :(xlr8r13_end)
xlr8r13 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( V '' . *add(xname,'V')
+                    | I '' . *add(xname,'I')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r13_end
        xlr8r_expressions()

True  y*x         V + '*' + V
True  z           V
True  -y          '-' + V
True  z           V
True  z/3         V + '/' + I
True  z           V
True  10-10*+x+z  I + '-' + I + '*' + '+' + V + '+' + V
True  6/z         I + '/' + V


Keep rerunning XLR8R. Now look for some smaller expression-pattern alternatives to inject back into XLR8R, some choices could be:

* I
* V
* σ('-') + V
* I + σ('*') + V
* V + σ('+') + I

We don't want to choose I or V patterns yet, since these could obscure results. We want less low-level constructs. After confirming by multiple XLR8R runs, that the following patterns are listed as proper alternatives each matching a FULL expression, then let's choose these next four new patterns to inject:

* V + ANY('+-*/') + V
* V + ANY('+-*/') + I
* I + ANY('+-*/') + V
* I + ANY('+-*/') + I

In [8]:
        V = ANY(lcase)
        I = SPAN(digits)
        X = V ANY('+-*/') V | V ANY('+-*/') I | I ANY('+-*/') V | I ANY('+-*/') I
        DEFINE('xlr8r(name)','xlr8r15')                        :(xlr8r15_end)
xlr8r15 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( X '' . *add(xname,'X')
+                    | V '' . *add(xname,'V')
+                    | I '' . *add(xname,'I')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r15_end
        xlr8r_expressions()

True  --10        '-' + '-' + I
True  y           V
True  7           I
True  16*((13))   I + '*' + '(' + '(' + I + ')' + ')'
True  3/7         X
True  ((3))       '(' + '(' + I + ')' + ')'
True  x+y         X
True  12*y-((9))  X + '-' + '(' + '(' + I + ')' + ')'


Keep running XLR8R. Several new smaller patterns are now emerging:

* X + σ('+') + V, X + σ('+') + I
* X + σ('-') + V, X + σ('-') + I
* X + σ('*') + V, X + σ('*') + I
* X + σ('/') + V, X + σ('/') + I

which is equivalent to:

* X + ANY('+-*/') + V
* X + ANY('+-*/') + I

So, let's choose these two new patterns to inject back into XLR8R.

In [9]:
        V = ANY(lcase)
        I = SPAN(digits)
        X = V ANY('+-*/') V | V ANY('+-*/') I | I ANY('+-*/') V | I ANY('+-*/') I
        DEFINE('xlr8r(name)','xlr8r17')                        :(xlr8r17_end)
xlr8r17 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( X '' . *add(xname,'X')
+                    | X ANY('+-*/') V '' . *add(xname,'X')
+                    | X ANY('+-*/') I '' . *add(xname,'X')
+                    | V '' . *add(xname,'V')
+                    | I '' . *add(xname,'I')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r17_end
        xlr8r_expressions()

True  x-(z)       V + '-' + '(' + V + ')'
True  (y-16)/1    '(' + X + ')' + '/' + I
True  12          I
True  z           V
True  z+10        X
True  13/11+y+z   X + '+' + X
True  10-2/y-15   X + '/' + X
True  y-(-6)/y    V + '-' + '(' + '-' + I + ')' + '/' + V


Keep running XLR8R. This looks promising, so let's inject these two patterns into pattern X. However, two problems arise when we attempt to do so. Firstly, the two patterns reference the X pattern, and when defining X, the variable X itself does not yet exist. Secondly, keeping X on the left versus the right of the operator will cause X to spin on an infinite loop. We solve the first problem, an undefined reference, by utilizing the unevaluated pattern expression, ζ("X"). We solve the second problem, an infinite loop, by swapping the X over to the right side of the operator since likely they are grammatically equivalent in this case.

In [10]:
        V = ANY(lcase)
        I = SPAN(digits)
        X = V ANY('+-*/') *X | I ANY('+-*/') *X | V ANY('+-*/') V | V ANY('+-*/') I
+         | I ANY('+-*/') V | I ANY('+-*/') I
        DEFINE('xlr8r(name)','xlr8r19')                        :(xlr8r19_end)
xlr8r19 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( X '' . *add(xname,'X')
+                    | V '' . *add(xname,'V')
+                    | I '' . *add(xname,'I')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r19_end
        xlr8r_expressions()

True  x+x         X
True  16          I
True  7           I
True  6           I
True  10          I
True  14/z        X
True  z           V
True  y           V


Keep running XLR8R. It is beginning to appear that within the grammar, both V and I behave syntactically the same. Knowing, of course, that these are calculator expressions, we could probably assume that the lowercase letters somehow also represent numbers; maybe memory register names. So let's merge these into a new pattern N, for number. Notice also that N alone is well-formed expression, so we inject N into the X pattern. And then simplify and refactor.

In [11]:
        V = ANY(lcase)
        I = SPAN(digits)
        N = V | I
        X = N ANY('+-*/') *X | N
        DEFINE('xlr8r(name)','xlr8r21')                        :(xlr8r21_end)
xlr8r21 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( X '' . *add(xname,'X')
+                    | N '' . *add(xname,'N')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r21_end
        xlr8r_expressions()

True  11          X
True  y+1         X
True  y           X
True  y           X
True  13          X
True  (y)         '(' + X + ')'
True  x+x         X
True  11          X


Keep running XRL8R. Notice two new patterns emerging:

* σ('+') + X
* σ('-') + X

So now injecting these into pattern X.

In [12]:
        V = ANY(lcase)
        I = SPAN(digits)
        N = V | I
        X = N ANY('+-*/') *X | ANY('+-') *X | N
        DEFINE('xlr8r(name)','xlr8r23')                        :(xlr8r23_end)
xlr8r23 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( X '' . *add(xname,'X')
+                    | N '' . *add(xname,'N')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r23_end
        xlr8r_expressions()

True  (15)        '(' + X + ')'
True  +z+z        X
True  (5++z)/6+y  '(' + X + ')' + '/' + X
True  x           X
True  (9)         '(' + X + ')'
True  y           X
True  y           X
True  z           X


Keep running XLR8R. Notice one new pattern emerging:

* σ('(') + X + σ(')')

So let's inject that into pattern X.

In [13]:
        V = ANY(lcase)
        I = SPAN(digits)
        N = V | I
        X = N ANY('+-*/') *X | ANY('+-') *X | N | '(' *X ')'
        DEFINE('xlr8r(name)','xlr8r25')                        :(xlr8r25_end)
xlr8r25 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( X '' . *add(xname,'X')
+                    | N '' . *add(xname,'N')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r25_end
        xlr8r_expressions()

True  3-x/z       X
True  -+4-6       X
True  15/y*1      X
True  16          X
True  (14*8+(0))  X
True  4-z++x      X
True  -16*x+(z)   X
True  y           X


Keep running XLR8R. Notice some very prevalent patterns emerging:

1. X + X
2. X + σ('+') + X
3. X + σ('/') + X

The first pattern seems strange and incorrect, so we will not choose it. However, the second and third are indicating an extension.

* N + ANY('+-*/') + ζ("X")

should somehow be extended to

* ζ("X") + ANY('+-*/') + ζ("X")

But again, that would cause an infinite loop. So we inject this new emerging pattern by rearranging and refactoring as follows:

In [14]:
        V = ANY(lcase)
        I = SPAN(digits)
        N = V | I
        E = N | '(' *X ')'
        X = E ANY('+-*/') *X | ANY('+-') *X | E
        DEFINE('xlr8r(name)','xlr8r27')                        :(xlr8r27_end)
xlr8r27 xname = name
        xlr8r = POS(0) '' . *clear(xname)
+               ARBNO( X '' . *add(xname,'X')
+                    | E '' . *add(xname,'E')
+                    | N '' . *add(xname,'N')
+                    | LEN(1) . tx '' . *add(xname,"'" tx "'") )
+               RPOS(0)                                         :(RETURN)
xlr8r27_end
        xlr8r_expressions()

True  16/2-10     X
True  z           X
True  10/14       X
True  z-5         X
True  z           X
True  (5)         X
True  x           X
True  ((0/z))     X


Notice now there is now only one pattern being generated for all generated expressions. That is, of course, the pattern X. So it appears that we have a calculator-expression parser named X working, and can begin applying the new X pattern as needed. With just a few adjustments, the Python code for an expression parser is completed using only four pattern variables.

In [15]:
        V = ANY(lcase)
        I = SPAN(digits)
        E = V | I | '(' *X ')'
        X = E ANY('+-') *X | E ANY('*/') *X | ANY('+-') *X | E
        k = 0
s29     k = LT(k,16) k + 1                                      :F(e29)
        subject = rand_expression()
        subject POS(0) X RPOS(0)                                :F(f29)
        OUTPUT = 'True  ' subject                               :(s29)
f29     OUTPUT = 'False ' subject                               :(s29)
e29

True  (2-10*(z)-y)
True  4*x/(y/4)-y+z-1/0
True  (2)
True  z
True  z/3
True  -6
True  8
True  14/-(y-x)/x
True  (y/5)
True  -(15)
True  -8*z/6
True  (1+z)-x
True  9
True  (y)+6+16
True  13*z
True  5/(z)/z+-y


Now, decorate the pattern to calculate and print results.

In [16]:
        DEFINE('push(v)')                                       :(push_end)
push    sp = sp + 1
        S[sp] = v                                               :(RETURN)
push_end
        DEFINE('pop()')                                         :(pop_end)
pop     pop = S[sp]
        sp = sp - 1                                             :(RETURN)
pop_end
        DEFINE('a_start()')                                        :(a_start_end)
a_start S = TABLE()
        sp = 0
        a_start = .dummy                                  :(NRETURN)
a_start_end
        DEFINE('a_var()')                                          :(a_var_end)
a_var   push($n)
        a_var = .dummy                                    :(NRETURN)
a_var_end
        DEFINE('a_int()')                                          :(a_int_end)
a_int   push(+n)
        a_int = .dummy                                    :(NRETURN)
a_int_end
        DEFINE('a_add()')                                          :(a_add_end)
a_add   push(pop() + pop())
        a_add = .dummy                                    :(NRETURN)
a_add_end
        DEFINE('a_sub()')                                          :(a_sub_end)
a_sub   push(pop() - pop())
        a_sub = .dummy                                    :(NRETURN)
a_sub_end
        DEFINE('a_mul()')                                          :(a_mul_end)
a_mul   push(pop() * pop())
        a_mul = .dummy                                    :(NRETURN)
a_mul_end
        DEFINE('a_div()')                                          :(a_div_end)
a_div   push(pop() / pop())
        a_div = .dummy                                    :(NRETURN)
a_div_end
        DEFINE('a_neg()')                                          :(a_neg_end)
a_neg   push(-pop())
        a_neg = .dummy                                    :(NRETURN)
a_neg_end
        DEFINE('a_print()')                                        :(a_print_end)
a_print OUTPUT = pop()
        a_print = .dummy                                  :(NRETURN)
a_print_end
        V = ANY(lcase) . n '' . *a_var()
        I = SPAN(digits) . n '' . *a_int()
        E = V | I | '(' *X ')'
        X = E '+' *X '' . *a_add()
+         | E '-' *X '' . *a_sub()
+         | E '*' *X '' . *a_mul()
+         | E '/' *X '' . *a_div()
+         | '+' *X
+         | '-' *X '' . *a_neg()
+         | E
        C = POS(0) '' . *a_start() X '' . *a_print() RPOS(0)
        x = 1
        y = 2
        z = 3
        'x+y*z' C
        'x+(y*z)' C
        '(x+y)*z' C

7
7
9


You can decorate parts of the pattern with debug-tracing. The following traces everything. Note the PRINT function must return True to allow pattern matching to continue.

In [17]:
*       show(tag) prints the tag and the cursor each time the matcher tries that part;
*       $ OUTPUT prints a piece of text the moment it is matched.
        DEFINE('show(tag)')                                     :(show_end)
show    OUTPUT = tag pos                                        :(RETURN)
show_end
        V = @pos *show('V') ANY(lcase) $ OUTPUT
        I = @pos *show('I') SPAN(digits) $ OUTPUT
        E = @pos *show('E')
+           ( V | I
+           | @pos *show('(') '(' $ OUTPUT *X @pos *show(')') ')' $ OUTPUT )
        X = @pos *show('X')
+           ( E @pos *show('+-*/') ANY('+-*/') $ OUTPUT *X
+           | @pos *show('+-') ANY('+-') $ OUTPUT *X
+           | E )
        k = 0
s33     k = LT(k,3) k + 1                                       :F(e33)
        subject = rand_expression()
        subject POS(0) X RPOS(0)                                :F(f33)
        OUTPUT = 'True  ' subject                               :(s33)
f33     OUTPUT = 'False ' subject                               :(s33)
e33

X0
E0
V0
I0
8
+-*/1
-
X2
E2
V2
z
+-*/3
+
X4
E4
V4
I4
(4
(
X5
E5
V5
z
+-*/6
+
X7
E7
V7
z
+-*/8
/
X9
E9
V9
I9
(9
+-9
+
X10
E10
V10
I10
(10
(
X11
E11
V11
I11
3
+-*/12
(11
+-11
E11
V11
I11
3
)12
)
+-*/13
(11
+-10
E10
V10
I10
(10
(
X11
E11
V11
I11
3
+-*/12
(11
+-11
E11
V11
I11
3
)12
)
)13
)
+-*/14
/
X15
E15
V15
I15
(15
(
X16
E16
V16
I16
15
+-*/18
-
X19
E19
V19
x
+-*/20
/
X21
E21
V21
z
+-*/22
-
X23
E23
V23
y
+-*/24
*
X25
E25
V25
z
+-*/26
/
X27
E27
V27
I27
(27
(
X28
E28
V28
z
+-*/29
-
X30
E30
V30
I30
(30
(
X31
E31
V31
x
+-*/32
-
X33
E33
V33
I33
7
+-*/34
(33
+-33
E33
V33
I33
7
)34
)
+-*/35
(33
I31
(31
+-31
E31
V31
x
)32
I31
(31
+-30
E30
V30
I30
(30
(
X31
E31
V31
x
+-*/32
-
X33
E33
V33
I33
7
+-*/34
(33
+-33
E33
V33
I33
7
)34
)
)35
)
+-*/36
(33
I31
(31
+-31
E31
V31
x
)32
I31
(31
I28
(28
+-28
E28
V28
z
)29
I28
(28
+-27
E27
V27
I27
(27
(
X28
E28
V28
z
+-*/29
-
X30
E30
V30
I30
(30
(
X31
E31
V31
x
+-*/32
-
X33
E33
V33
I33
7
+-*/34
(33
+-33
E33
V33
I33
7
)34
)
+-*/35
(33
I31
(31
+-31
E31
V31
x
)32
I3

Alternatively, you can call TRACE(10), with 10, 20, 30 levels. Call TRACE(40) to revert back to normal error reporting.

In [18]:
*       SNOBOL4 has its own tracing (the keyword &TRACE and the function TRACE);
*       here the undecorated pattern is matched once more.
        V = ANY(lcase)
        I = SPAN(digits)
        E = V | I | '(' *X ')'
        X = E ANY('+-') *X | E ANY('*/') *X | ANY('+-') *X | E
        subject = rand_expression()
        subject POS(0) X RPOS(0)                                :F(f35)
        OUTPUT = 'True  ' subject                               :(e35)
f35     OUTPUT = 'False ' subject
e35

True  (y*1)
